In [23]:
import pandas as pd
import os
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import quote_plus
from rdflib.namespace import RDF, DC, Namespace
import xml.etree.ElementTree as ET
from lxml import etree
import shutil
import zipfile
import ftplib
import io
import csv
import field_extractor as fe

In [24]:
# Constants
UNZIP_DIR = "selected_data"
FTP_HOST = "download.europeana.eu"
FTP_PATH = "dataset/XML/"
OUTPUT_DIR = "collected_data"

In [33]:
data_ids = []
# extract the dataset_ids
with open('dataset_ids.txt', 'r') as f:
    data_ids = f.readlines()
    data_ids = [x.strip() for x in data_ids]

data_ids = data_ids[166]
print(data_ids)

375.zip


In [34]:
# Example processing: save the first element to a UNZIP_DIR as an xml file
if not os.path.exists(UNZIP_DIR):
    os.makedirs(UNZIP_DIR)

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)


In [27]:
def download_file(ftp_host, ftp_path, filename):
    zip_data = io.BytesIO()

    with ftplib.FTP(ftp_host) as ftp:
        ftp.login()  # Login as anonymous
        ftp.cwd(ftp_path)

        ftp.retrbinary(f'RETR {filename}', zip_data.write)
    
    zip_data.seek(0)
    return zip_data

# Function to unzip a file
def unzip_file(zip_data):
    extracted_files = []  # List to store file content

    with zipfile.ZipFile(zip_data, 'r') as zip_ref:
        for file_info in zip_ref.infolist():
            with zip_ref.open(file_info) as file:
                file_content = file.read()
                extracted_files.append(file_content)

    return extracted_files

# Function to download and process a ZIP file
def download_zip(filename):

    try:
        print(f"Starting download and processing for {filename}...")
        
        # Download the ZIP file into memory
        zip_data = download_file(FTP_HOST, FTP_PATH, filename)

        # Unzip and process the file
        extracted_files = unzip_file(zip_data)
        print(f"deleting zip file {filename}")
        del zip_data 

        # print the filenames from the extracted files
        print(f"Extracted files: {(extracted_files[0])}")

        # make a subdirectory for the dataset
        dataset_dir = os.path.join(UNZIP_DIR, filename)
        dataset_dir = dataset_dir[:-4]
        if not os.path.exists(dataset_dir):
            os.makedirs(dataset_dir)
        
        # save the extracted files in the subdirectory
        for i, file_content in enumerate(extracted_files):
            with open(os.path.join(dataset_dir, f"{i}.xml"), 'wb') as file:
                file.write(file_content)

        print(f"Finished processing {filename}")
        del extracted_files

    except Exception as e:
        print(f"Error processing {filename}: {e}")

# for single dataset

# for multiple datasets

In [28]:
# Use ThreadPoolExecutor to download and process files in parallel
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = [executor.submit(download_zip, filename) for filename in data_ids]

    # Use tqdm to show progress as futures are completed
    for future in tqdm(as_completed(futures), total=len(futures), desc="Processing ZIP files"):
        # This will raise exceptions if any occurred during processing
        future.result()

Starting download and processing for 9200249.zip...
Starting download and processing for 2064129.zip...
Starting download and processing for 15416.zip...
Starting download and processing for 574.zip...
Starting download and processing for 92030.zip...


Processing ZIP files:   0%|          | 0/5 [00:00<?, ?it/s]

deleting zip file 15416.zip
Extracted files: b'<?xml version="1.0" encoding="UTF-8" standalone="yes"?><rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#" xmlns:dc="http://purl.org/dc/elements/1.1/" xmlns:dcterms="http://purl.org/dc/terms/" xmlns:edm="http://www.europeana.eu/schemas/edm/" xmlns:owl="http://www.w3.org/2002/07/owl#" xmlns:wgs84_pos="http://www.w3.org/2003/01/geo/wgs84_pos#" xmlns:skos="http://www.w3.org/2004/02/skos/core#" xmlns:rdaGr2="http://rdvocab.info/ElementsGr2/" xmlns:foaf="http://xmlns.com/foaf/0.1/" xmlns:ebucore="http://www.ebu.ch/metadata/ontologies/ebucore/ebucore#" xmlns:doap="http://usefulinc.com/ns/doap#" xmlns:odrl="http://www.w3.org/ns/odrl/2/" xmlns:cc="http://creativecommons.org/ns#" xmlns:ore="http://www.openarchives.org/ore/terms/" xmlns:svcs="http://rdfs.org/sioc/services#" xmlns:oa="http://www.w3.org/ns/oa#" xmlns:dqv="http://www.w3.org/ns/dqv#"><edm:ProvidedCHO rdf:about="http://data.europeana.eu/item/15416/Data_Library3_Library3_Batc

Processing ZIP files: 100%|██████████| 5/5 [00:01<00:00,  4.15it/s]

deleting zip file 2064129.zip
Extracted files: b'<?xml version="1.0" encoding="UTF-8" standalone="yes"?><rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#" xmlns:dc="http://purl.org/dc/elements/1.1/" xmlns:dcterms="http://purl.org/dc/terms/" xmlns:edm="http://www.europeana.eu/schemas/edm/" xmlns:owl="http://www.w3.org/2002/07/owl#" xmlns:wgs84_pos="http://www.w3.org/2003/01/geo/wgs84_pos#" xmlns:skos="http://www.w3.org/2004/02/skos/core#" xmlns:rdaGr2="http://rdvocab.info/ElementsGr2/" xmlns:foaf="http://xmlns.com/foaf/0.1/" xmlns:ebucore="http://www.ebu.ch/metadata/ontologies/ebucore/ebucore#" xmlns:doap="http://usefulinc.com/ns/doap#" xmlns:odrl="http://www.w3.org/ns/odrl/2/" xmlns:cc="http://creativecommons.org/ns#" xmlns:ore="http://www.openarchives.org/ore/terms/" xmlns:svcs="http://rdfs.org/sioc/services#" xmlns:oa="http://www.w3.org/ns/oa#" xmlns:dqv="http://www.w3.org/ns/dqv#"><edm:ProvidedCHO rdf:about="http://data.europeana.eu/item/2064129/Museu_ProvidedCHO_museu

In [29]:
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# Function to process a single subdirectory
def process_single_subdirectory(subdirectory_path):
    try:
        subdirectory = os.path.basename(subdirectory_path)
        print(f"Processing {subdirectory}...")

        # Get the size of the subdirectory
        size = len(os.listdir(subdirectory_path))
        print(f"Size of {subdirectory}: {size}")
        print(subdirectory_path)

        # Process RDF files in the subdirectory
        output = fe.parse_rdf_files(subdirectory_path)

        # Write the output to an XML file
        output_file = f'/home/sbasir/Thesis/Thesis/EDP/collected_data/{subdirectory}.xml'
        fe.write_data(output, output_file)

        print(f"Finished processing {subdirectory}. Output written to {output_file}")

    except Exception as e:
        print(f"Error processing {subdirectory}: {e}")

# Threaded function to process all subdirectories in the given directory
def process_zip_threaded(directory):
    subdirectories = [os.path.join(directory, subdir) for subdir in os.listdir(directory) if os.path.isdir(os.path.join(directory, subdir))]

    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = {executor.submit(process_single_subdirectory, subdir): subdir for subdir in subdirectories}

        # Use tqdm to show progress as futures are completed
        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing subdirectories"):
            try:
                future.result()
            except Exception as e:
                subdirectory = futures[future]
                print(f"Error processing subdirectory {subdirectory}: {e}")

# Example usage
process_zip_threaded('/home/sbasir/Thesis/Thesis/EDP/selected_data')

Processing 574...
Processing 2064129...
Processing 92030...
Processing 15416...
Processing 9200249...


Processing subdirectories:   0%|          | 0/5 [00:00<?, ?it/s]

Size of 574: 1567
/home/sbasir/Thesis/Thesis/EDP/selected_data/574
Size of 2064129: 1573
/home/sbasir/Thesis/Thesis/EDP/selected_data/2064129
Size of 15416: 1568
/home/sbasir/Thesis/Thesis/EDP/selected_data/15416
Size of 9200249: 1574
/home/sbasir/Thesis/Thesis/EDP/selected_data/9200249
Size of 92030: 1560
/home/sbasir/Thesis/Thesis/EDP/selected_data/92030


Processing subdirectories:  20%|██        | 1/5 [00:24<01:39, 24.96s/it]

Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/9200249.xml
Finished processing 9200249. Output written to /home/sbasir/Thesis/Thesis/EDP/collected_data/9200249.xml


...

Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/15416.xml
Finished processing 15416. Output written to /home/sbasir/Thesis/Thesis/EDP/collected_data/15416.xml
Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/92030.xml
Finished processing 92030. Output written to /home/sbasir/Thesis/Thesis/EDP/collected_data/92030.xml


Processing subdirectories:  80%|████████  | 4/5 [00:25<00:04,  4.01s/it]

Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/574.xml
Finished processing 574. Output written to /home/sbasir/Thesis/Thesis/EDP/collected_data/574.xml


Processing subdirectories: 100%|██████████| 5/5 [00:26<00:00,  5.28s/it]

Data written to /home/sbasir/Thesis/Thesis/EDP/collected_data/2064129.xml
Finished processing 2064129. Output written to /home/sbasir/Thesis/Thesis/EDP/collected_data/2064129.xml


In [30]:
# def process_zip(directory):
#     # for each subdirectory in the UNZIP_DIR print the size of the subdirectory
#     for subdirectory in os.listdir(directory):
#         subdirectory_path = os.path.join(directory, subdirectory)
#         print(f"Size of {subdirectory}: {len(os.listdir(subdirectory_path))}")
#         print(subdirectory_path)
#         output = fe.parse_rdf_files(subdirectory_path)
#         fe.write_data(output, f'/home/sbasir/Thesis/Thesis/EDP/collected_data/{subdirectory}.xml')

# process_zip('/home/sbasir/Thesis/Thesis/EDP/selected_data')

In [31]:
# # Usage
# directory = '/home/sbasir/Thesis/Thesis/EDP/selected_data/254'  # Change this to your directory containing RDF/XML files
# solr_xml_data = fe.parse_rdf_files(directory)
# # solr_xml_data = fe.remove_duplicates(solr_xml_data)
# output_file = '/home/sbasir/Thesis/Thesis/EDP/testing.xml'
# fe.write_data(solr_xml_data, output_file)